In [1]:
import numpy as np
import matplotlib as plt
import seaborn as sea
import pandas as pd
import xgboost as xgb
import sklearn as sk
import sqlite3 
import mysql.connector
import duckdb # for me to use sql even though it is not necessary with the way the datasets are given

## Data Definition

I am asked by Northern Lights Air (NLA) to run an analysis on their loyalty program. NLA ran a promotional campaign from Februrary 2018 (2018-02-01) until April 2018 (2018-04-31). NLA wants to improve their loyalty program, hence they want to know:
- What impact did the campaign have on the loyalty memberships? (Enrollment, Cancellation, Points redeemed etc.)
- What impact did the campaign have on booked flights? (in the summer, in general etc.)
- What impact did it have on customer churning?
- Was the campaign more successful towards certain demographics? (Gender, Education, Marital Status etc.)

## Exploratory Analysis

I will use SQL (for fun) to explore the data. 

**392936** total data entries index starting at 0

### Calendar 
This is a dataset that gives me everyday from 2012-01-01 (January 1, 2012) until 2018-12-31 (December 31, 2018). It gives me:
- Start of Year: first date of the year
- Start of Quarter: first date of the quarter it's in (Jan 1, Apr 1, Jul 1, Oct 1)
- Start of Month: first date of the month it's in

### Customer Flights Activity 
- Loyalty Number: **16737** unique members based on loyalty numbers
- Year
- Month
- Total Flights: Do not have to be loyalty member to fly
- Distance
- Points Accumulated
- Points Redeemed
- Dollar Cost Points Redeemed (CAD)

### Loyalty Program 
- Loyalty Number: **16737** unique members based on loyalty numbers
- Country: Canada (constant)
- Province
- City
- Postal Code
- Gender
- Education
- Salary (CAD): Contains NULL
- Marital Status
- Loyalty Card
- CLV
- Enrollment Type
- Enrollment Year
- Enrollment Month
- Cancellation Year: Contains NULL
- Cancellation Month: Contains NULL

### New Column Ideas
- Enrollment_length: Amount of time you've been in the program

I can see that we can **combine** Enrollment Year and Month together into one column, as well as Cancellation Year and Month. \
Can **combine** the two tables via Loyalty Number. \
Make two different tables, one that matches enrolment date to the calendar and one that matches cancellation date to calendar.\
Notice that column names are strings let's change that


In [85]:
#Quick check to see the tables
calendar_df = pd.read_csv("Calendar.csv")
flight_df = pd.read_csv("Customer Flight Activity.csv")
loyalty_df = pd.read_csv("Customer Loyalty History.csv")

# print(flight_df.head())
# print(loyalty_df.head())

In [ ]:
# #double checking cell 
# duckdb.sql(""" 
#     SELECT
#         cfa."Loyalty Number",
#         Year,
#         "Cancellation Year"
#     FROM read_csv_auto('Customer Flight Activity.csv') cfa
#     LEFT JOIN read_csv_auto('Customer Loyalty History.csv') clh
#         on cfa."Loyalty Number" = clh."Loyalty Number"
#     WHERE "Cancellation Year" < "Year"
#            """).df()

# Check for null valuesabs
duckdb.sql("""
SELECT 
    column_name,
    column_type,
    null_percentage
FROM(SUMMARIZE(
        SELECT *
        FROM read_csv_auto('Customer Flight Activity.csv') cfa
        LEFT JOIN read_csv_auto('Customer Loyalty History.csv') clh
            on cfa."Loyalty Number" = clh."Loyalty Number"))
""").df()

#87% of people have not churned away from the loyalty programabs
#25% null values 

In [83]:
#to avoid repetition we can aggregate the stats of each distinct loyalty number via summation
#assign it to dataframe called "flight"
flight = duckdb.sql("""
SELECT 
    "Loyalty Number",
    SUM("Total Flights") AS "Total Flights",
    SUM(Distance) AS "Total Distance",
    SUM("Points Accumulated") AS "Total Points",
    SUM("Points Redeemed") AS "Total Redeemed Points",
    ROUND("Total Redeemed Points" / "Total Points", 3)*100 AS "Percentage Redeemed (%)"
FROM flight_df 
GROUP BY "Loyalty Number" 
ORDER BY "Loyalty Number"
""").df()

loyalty = duckdb.sql("""
SELECT * EXCLUDE("Postal Code", "City", "Country", "Enrollment Year", "Enrollment Month", "Cancellation Year", "Cancellation Month"),
    MAKE_DATE("Enrollment Year", "Enrollment Month", 1) AS "Enrollment Date",
    MAKE_DATE("Cancellation Year", "Cancellation Month", 1) AS "Cancellation Date",
    CASE WHEN ("Cancellation Year" AND "Cancellation Month") IS NULL THEN 0 ELSE 1 END AS Cancelled,
    CASE WHEN Salary IS NULL THEN 1 ELSE 0 END AS Salary_Missing
FROM read_csv_auto("Customer Loyalty History.csv") 
""").df()

### Due Diligence Data Clean and Check

- Null values
- Correct data types
- Correct formatting for datetime
- check multicollinearity
- check data variance

In [89]:
#Join tables 
#Concat Enrollment and Cancellation Month and Year together
#We use day as 01 
#Cancelled is the target (Y)
main_df = duckdb.sql(""" 
    SELECT * 
    FROM loyalty l
    LEFT JOIN flight f USING ("Loyalty Number")
    ORDER BY l."Loyalty Number"
           """).df()
main_df.head()

,Loyalty Number,Province,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Date,Cancellation Date,Cancelled,Salary_Missing,Total Flights,Total Distance,Total Points,Total Redeemed Points,Percentage Redeemed (%)
0,100018,Alberta,Female,Bachelor,92552,Married,Aurora,7919.20,Standard,2016-08-01,NaT,0,0,46.0,81190.0,81190.0,1513.0,1.9
1,100102,Ontario,Male,College,<NA>,Single,Nova,2887.74,Standard,2013-03-01,NaT,0,1,51.0,68918.0,68918.0,1195.0,1.7
2,100140,British Columbia,Female,College,<NA>,Divorced,Nova,2838.07,Standard,2016-07-01,NaT,0,1,47.0,72856.0,72856.0,593.0,0.8
3,100214,British Columbia,Male,Bachelor,63253,Married,Star,4170.57,Standard,2015-08-01,NaT,0,0,22.0,38236.0,38236.0,861.0,2.3
4,100272,Ontario,Female,Bachelor,91163,Divorced,Star,6622.05,Standard,2014-01-01,NaT,0,0,37.0,54997.0,54997.0,1007.0,1.8


## Decision Tree Analysis

- `main_df` is the df we will be using that has joined tables between loyalty history and flight activity
- 

In [90]:
main_df.head()

,Loyalty Number,Province,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Date,Cancellation Date,Cancelled,Salary_Missing,Total Flights,Total Distance,Total Points,Total Redeemed Points,Percentage Redeemed (%)
0,100018,Alberta,Female,Bachelor,92552,Married,Aurora,7919.20,Standard,2016-08-01,NaT,0,0,46.0,81190.0,81190.0,1513.0,1.9
1,100102,Ontario,Male,College,<NA>,Single,Nova,2887.74,Standard,2013-03-01,NaT,0,1,51.0,68918.0,68918.0,1195.0,1.7
2,100140,British Columbia,Female,College,<NA>,Divorced,Nova,2838.07,Standard,2016-07-01,NaT,0,1,47.0,72856.0,72856.0,593.0,0.8
3,100214,British Columbia,Male,Bachelor,63253,Married,Star,4170.57,Standard,2015-08-01,NaT,0,0,22.0,38236.0,38236.0,861.0,2.3
4,100272,Ontario,Female,Bachelor,91163,Divorced,Star,6622.05,Standard,2014-01-01,NaT,0,0,37.0,54997.0,54997.0,1007.0,1.8
